In [2]:
import sys
import subprocess
from pathlib import Path

In [3]:
WORKDIR = Path("./_workspace")
WORKDIR.mkdir(exist_ok=True)

In [4]:
# krok 1
#implementacja
tokenizer_code = '''
import re

class Tokenizer:
    """Konfigurowany tokenizator: HTML strip + case + min length filter."""

    def __init__(self, lower: bool = True, strip_html: bool = True, min_length: int = 1):
        self.lower = lower
        self.strip_html = strip_html
        self.min_length = min_length

    def tokenize(self, text: str) -> list[str]:
        if self.strip_html:
            text = re.sub(r"<[^>]+>", " ", text)

        if self.lower:
            text = text.lower()

        tokens = re.findall(r"\\w+", text, flags=re.UNICODE)

        return [token for token in tokens if len(token) >= self.min_length]

    def vocab(self, texts: list[str]) -> set[str]:
        all_tokens = set()

        for text in texts:
            all_tokens.update(self.tokenize(text))

        return all_tokens
'''

In [5]:
#krok 2
#testy pytest
tests_code = '''
import pytest
from tokenizer import Tokenizer

@pytest.fixture
def tokenizer():
    return Tokenizer()

@pytest.fixture
def imdb_sample():
    from datasets import load_dataset
    ds = load_dataset("stanfordnlp/imdb", split="train").shuffle(seed=42).select(range(20))
    return [r["text"] for r in ds]

@pytest.mark.parametrize("text, expected_len", [
    ("", 0),
    ("<br><p></p>", 0),
    ("Hello WORLD!", 2),
    ("...!?!?!?", 0),
    ("zażółć gęślą jaźń", 3),
    ("the cat sat on the mat", 6),
])
def test_tokenize_cases(tokenizer, text, expected_len):
    assert len(tokenizer.tokenize(text)) == expected_len

def test_html_strip():
    assert Tokenizer().tokenize("<br>Hello WORLD!") == ["hello", "world"]

def test_without_lowercase():
    assert Tokenizer(lower=False).tokenize("Hello") == ["Hello"]

def test_without_html_strip():
    assert Tokenizer(strip_html=False).tokenize("<br>hello") == ["br", "hello"]

def test_vocab_dedup(tokenizer):
    assert tokenizer.vocab(["aa bb", "bb cc"]) == {"aa", "bb", "cc"}

def test_min_length_filter():
    tok = Tokenizer(min_length=4)
    assert tok.tokenize("a bb ccc dddd eeeee") == ["dddd", "eeeee"]

def test_imdb_integration(tokenizer, imdb_sample):
    vocab = tokenizer.vocab(imdb_sample)
    assert len(vocab) > 500, f"za malo unikalnych tokenow: {len(vocab)}"

@pytest.mark.xfail(reason="Tokenizer nie traktuje adresu e-mail jako jednego tokena")
def test_advanced_regex_unsupported():
    tok = Tokenizer()
    assert tok.tokenize("user@domain.com")[0] == "user@domain.com"
'''

In [7]:
#krok 3
#zapis plików
(WORKDIR / "tokenizer.py").write_text(tokenizer_code, encoding="utf-8")
(WORKDIR / "test_tokenizer.py").write_text(tests_code, encoding="utf-8")

1543

In [8]:
#krok 4
#uruchomienie pytest
result = subprocess.run(
    [sys.executable, "-m", "pytest", "test_tokenizer.py", "-v", "--tb=short"],
    capture_output=True,
    text=True,
    cwd=str(WORKDIR)
)

print("STDOUT:")
print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

STDOUT:
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/_workspace
plugins: typeguard-4.5.2, anyio-4.13.0, langsmith-0.8.15
collecting ... collected 13 items

test_tokenizer.py::test_tokenize_cases[-0] PASSED                        [  7%]
test_tokenizer.py::test_tokenize_cases[<br><p></p>-0] PASSED             [ 15%]
test_tokenizer.py::test_tokenize_cases[Hello WORLD!-2] PASSED            [ 23%]
test_tokenizer.py::test_tokenize_cases[...!?!?!?-0] PASSED               [ 30%]
test_tokenizer.py::test_tokenize_cases[za\u017c\xf3\u0142\u0107 g\u0119\u015bl\u0105 ja\u017a\u0144-3] PASSED [ 38%]
test_tokenizer.py::test_tokenize_cases[the cat sat on the mat-6] PASSED  [ 46%]
test_tokenizer.py::test_html_strip PASSED                                [ 53%]
test_tokenizer.py::test_without_lowercase PASSED                         [ 61%]
test_tok

In [9]:
#insight
sys.path.insert(0, str(WORKDIR))

from tokenizer import Tokenizer
from datasets import load_dataset

tokenizer = Tokenizer()

ds_100 = load_dataset("stanfordnlp/imdb", split="train").shuffle(seed=42).select(range(100))
texts_100 = [r["text"] for r in ds_100]

vocab_100 = tokenizer.vocab(texts_100)
avg_unique_per_review = len(vocab_100) / 100

In [10]:
print("\nInsight:")
print(f"Liczba unikalnych tokenów dla 100 recenzji: {len(vocab_100)}")
print(f"Średnio unikalnych tokenów na 1 recenzję: {avg_unique_per_review:.2f}")


Insight:
Liczba unikalnych tokenów dla 100 recenzji: 5053
Średnio unikalnych tokenów na 1 recenzję: 50.53


### Wnioski

Testy zostały wykonane poprawnie — 12 testów zakończyło się sukcesem, a 1 test został oznaczony jako `xfail`, czyli oczekiwane niepowodzenie. Oznacza to, że podstawowa implementacja klasy `Tokenizer` działa zgodnie z wymaganiami: usuwa HTML, zamienia tekst na małe litery, obsługuje polskie znaki, filtruje tokeny po długości i poprawnie buduje słownik unikalnych tokenów.

Test oznaczony jako `xfail` dotyczył traktowania adresu e-mail jako jednego tokena. Tokenizer nie obsługuje takiego przypadku, ale jest to świadomie zaznaczone w testach, więc nie jest traktowane jako błąd implementacji.

Dla 100 recenzji IMDB uzyskano 5053 unikalne tokeny, czyli średnio około 50.53 unikalnych tokenów na jedną recenzję. Pokazuje to, że nawet przy niewielkiej próbce tekstów słownik rośnie dość szybko, co jest typowe dla danych tekstowych.